In [0]:
%sql

INSERT OVERWRITE proyecto_final.silver.comunas (
  comuna_id,
  tipo_entidad,
  barrios,
  perimetro_m,
  area_m2,
  coordenadas
)
WITH datos_limpios AS (
  SELECT 
    comuna AS comuna_id,
    LOWER(REPLACE(TRIM(objeto), '"', '')) AS tipo_entidad,
    LOWER(REPLACE(TRIM(barrios), '"', '')) AS barrios,
    ROUND(perimetro, 2) AS perimetro_m,
    ROUND(area, 2) AS area_m2,
    geometry AS coordenadas
  FROM proyecto_final.raw.comunas_bronze
  WHERE comuna IS NOT NULL 
  AND geometry IS NOT NULL
),
datos_deduplicados AS (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY comuna_id ORDER BY comuna_id ) AS rn
  FROM datos_limpios
)
SELECT 
  comuna_id,
  tipo_entidad,
  barrios,
  perimetro_m,
  area_m2,
  coordenadas
FROM datos_deduplicados
WHERE rn = 1;